In [27]:
import torch
from torch import nn

In [28]:
# Cell 2


class LinearRegression(nn.Module):

    def __init__(
        self,
        num_inputs,
        learning_rate,
    ):

        # nn.Module의 parameter 관리 기능 초기화
        super().__init__()

        # learning rate: Step size
        self.learning_rate = learning_rate

        # Weight shape: [1, num_inputs]
        # Bias shape:   [1,]
        self.net = nn.Linear(
            in_features=num_inputs,
            out_features=1,
        )

        # 작은 난수로 weight 초기화
        nn.init.normal_(
            self.net.weight,
            mean=0.0,
            std=0.01,
        )

        # bias는 0으로 초기화
        nn.init.zeros_(self.net.bias)

    def forward(self, features):

        # 내부적으로 다음 계산을 수행한다.
        # features @ self.net.weight.T + self.net.bias
        return self.net(features)

In [29]:
model = LinearRegression(
    num_inputs=2,
    learning_rate=0.03,
)

print(model)

print("\nWeight:")
print(model.net.weight)
print("Weight shape:", model.net.weight.shape)

print("\nBias:")
print(model.net.bias)
print("Bias shape:", model.net.bias.shape)

LinearRegression(
  (net): Linear(in_features=2, out_features=1, bias=True)
)

Weight:
Parameter containing:
tensor([[-0.0004,  0.0189]], requires_grad=True)
Weight shape: torch.Size([1, 2])

Bias:
Parameter containing:
tensor([0.], requires_grad=True)
Bias shape: torch.Size([1])


In [ ]:
features = torch.tensor(
    [
        [1.0, 2.0],
        [2.0, 1.0],
        [3.0, 4.0],
        [4.0, 3.0],
    ]
)

# model(features)를 호출하면
# nn.Module.__call__을 거쳐 forward(features)가 실행
predictions = model(features)

print("Features shape:", features.shape)
print("Predictions:")
print(predictions)
print("Predictions shape:", predictions.shape)

Features shape: torch.Size([4, 2])
Predictions:
tensor([[0.0374],
        [0.0181],
        [0.0744],
        [0.0551]], grad_fn=<AddmmBackward0>)
Predictions shape: torch.Size([4, 1])


In [ ]:
# nn.Linear 내부 계산과 직접 계산이 같은지 확인
manual_predictions = (
    features @ model.net.weight.T
    + model.net.bias
)

print("Model predictions:")
print(predictions)

print("\nManual predictions:")
print(manual_predictions)

assert torch.allclose(
    predictions,
    manual_predictions,
)

Model predictions:
tensor([[0.0374],
        [0.0181],
        [0.0744],
        [0.0551]], grad_fn=<AddmmBackward0>)

Manual predictions:
tensor([[0.0374],
        [0.0181],
        [0.0744],
        [0.0551]], grad_fn=<AddBackward0>)


In [32]:
loss_function = nn.MSELoss(
    reduction="mean",
)

labels = torch.tensor(
    [
        [1.0],
        [6.0],
        [-1.0],
        [4.0],
    ]
)

# 현재 모델 parameter로 예측한다.
predictions = model(features)

# 예측값과 label의 shape를 확실히 맞춘다.
labels = labels.reshape(predictions.shape)

# 모든 sample의 squared error를 평균낸 scalar loss
loss = loss_function(
    predictions,
    labels,
)

print("Predictions:")
print(predictions)

print("\nLabels:")
print(labels)

print("\nMSE loss:")
print(loss)

print("Loss shape:", loss.shape)

Predictions:
tensor([[0.0374],
        [0.0181],
        [0.0744],
        [0.0551]], grad_fn=<AddmmBackward0>)

Labels:
tensor([[ 1.],
        [ 6.],
        [-1.],
        [ 4.]])

MSE loss:
tensor(13.3566, grad_fn=<MseLossBackward0>)
Loss shape: torch.Size([])


In [33]:
# nn.MSELoss의 계산을 직접 확인한다.
errors = predictions - labels

manual_mse = errors.pow(2).mean()
scratch_style_loss = 0.5 * errors.pow(2).mean()

print("nn.MSELoss:", loss.item())
print("Manual MSE:", manual_mse.item())
print("Scratch-style loss:", scratch_style_loss.item())

nn.MSELoss: 13.356607437133789
Manual MSE: 13.356607437133789
Scratch-style loss: 6.6783037185668945


In [34]:
# model.parameters()가 등록된 weight와 bias를 제공
parameters = list(model.parameters())

print("Number of parameter tensors:", len(parameters))

print("\nWeight parameter shape:")
print(parameters[0].shape)

print("\nBias parameter shape:")
print(parameters[1].shape)

Number of parameter tensors: 2

Weight parameter shape:
torch.Size([1, 2])

Bias parameter shape:
torch.Size([1])


In [35]:
# SGD instance를 생성할 때는 다음 두 가지를 지정한다.
# 1. 최적화할 parameter
# 2. 최적화 알고리즘에서 사용할 learning rate

# PyTorch가 구현한 SGD optimizer를 생성한다.
optimizer = torch.optim.SGD(
    params=model.parameters(),
    lr=model.learning_rate,
)

print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.03
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [ ]:
# PyTorch optimizer를 사용하여 parameter를 한 번 갱신한다.

# 이전 backward에서 남은 gradient를 제거
optimizer.zero_grad()

# 현재 parameter로 예측값을 다시 계산
predictions = model(features)

# 현재 예측값으로 새로운 계산 그래프와 loss를 만든다.
loss = loss_function(
    predictions,
    labels,
)

# 갱신 전 parameter를 복사
weights_before_update = model.net.weight.detach().clone()
bias_before_update = model.net.bias.detach().clone()


# Weight와 bias의 gradient를 계산
loss.backward()

print("Weight gradient:")
print(model.net.weight.grad)

print("\nBias gradient:")
print(model.net.bias.grad)


# 계산된 gradient로 parameter를 실제 갱신
optimizer.step()

print("\nWeights before update:")
print(weights_before_update)

print("\nWeights after update:")
print(model.net.weight)

print("\nBias before update:")
print(bias_before_update)

print("\nBias after update:")
print(model.net.bias)

Weight gradient:
tensor([[-12.7413,  -7.7220]])

Bias gradient:
tensor([-4.9075])

Weights before update:
tensor([[-0.0004,  0.0189]])

Weights after update:
Parameter containing:
tensor([[0.3818, 0.2506]], requires_grad=True)

Bias before update:
tensor([0.])

Bias after update:
Parameter containing:
tensor([0.1472], requires_grad=True)


In [37]:
# Cell 11
from torch.utils.data import DataLoader, TensorDataset

true_weights = torch.tensor([2.0, -3.4])
true_bias = 4.2

number_of_examples = 1000
number_of_features = 2
batch_size = 32

# X.shape = (1000, 2)
training_features = torch.randn(
    number_of_examples,
    number_of_features,
)

# epsilon ~ N(0, 0.01^2)
noise = torch.randn(number_of_examples, 1) * 0.01

# y = Xw + b + epsilon
# y.shape = (1000, 1)
training_labels = training_features @ true_weights.reshape(-1, 1) + true_bias + noise

training_dataset = TensorDataset(
    training_features,
    training_labels,
)

training_loader = DataLoader(
    dataset=training_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

print("Features shape:", training_features.shape)
print("Labels shape:", training_labels.shape)
print("Number of batches:", len(training_loader))

assert training_features.shape == (1000, 2)
assert training_labels.shape == (1000, 1)

Features shape: torch.Size([1000, 2])
Labels shape: torch.Size([1000, 1])
Number of batches: 32


In [38]:
# Cell 12
# 훈련용 모델을 새로 생성한다.
model = LinearRegression(
    num_inputs=2,
    learning_rate=0.03,
)

# PyTorch의 평균제곱오차
loss_function = nn.MSELoss(
    reduction="mean",
)

# PyTorch의 SGD optimizer
optimizer = torch.optim.SGD(
    params=model.parameters(),
    lr=model.learning_rate,
)

number_of_epochs = 10
loss_history = []

# Dropout이나 BatchNorm이 있다면 훈련 방식으로 동작하게 한다.
# 현재 선형층의 결과에는 영향을 주지 않지만 좋은 습관이다.
model.train()

for epoch in range(number_of_epochs):
    total_loss = 0.0
    total_samples = 0

    for batch_features, batch_labels in training_loader:
        # 1. 이전 batch의 gradient를 제거한다.
        optimizer.zero_grad()

        # 2. 현재 batch의 예측값을 계산한다.
        predictions = model(batch_features)

        # 3. 현재 batch의 평균 MSE를 계산한다.
        loss = loss_function(
            predictions,
            batch_labels,
        )

        # 4. 모든 parameter의 gradient를 계산한다.
        loss.backward()

        # 5. Gradient를 이용해 parameter를 갱신한다.
        optimizer.step()

        current_batch_size = batch_labels.shape[0]

        # Epoch 전체의 평균 loss를 구하기 위해
        # batch 평균 loss에 sample 수를 다시 곱한다.
        total_loss += loss.item() * current_batch_size
        total_samples += current_batch_size

    epoch_loss = total_loss / total_samples
    loss_history.append(epoch_loss)

    print(f"Epoch {epoch + 1} " f"| Mean loss: {epoch_loss:.6f}")

Epoch 1 | Mean loss: 9.024181
Epoch 2 | Mean loss: 0.160235
Epoch 3 | Mean loss: 0.002958
Epoch 4 | Mean loss: 0.000153
Epoch 5 | Mean loss: 0.000099
Epoch 6 | Mean loss: 0.000097
Epoch 7 | Mean loss: 0.000097
Epoch 8 | Mean loss: 0.000097
Epoch 9 | Mean loss: 0.000097
Epoch 10 | Mean loss: 0.000098


In [39]:
# Cell 13
# 평가할 때는 evaluation mode로 전환한다.
model.eval()

# Parameter를 읽기만 하므로 gradient 기록을 중지한다.
with torch.no_grad():
    learned_weights = model.net.weight.reshape(true_weights.shape).clone()

    learned_bias = model.net.bias.clone()

weight_error = true_weights - learned_weights
bias_error = true_bias - learned_bias

print("True weights:", true_weights)
print("Learned weights:", learned_weights)
print("Weight error:", weight_error)

print("\nTrue bias:", true_bias)
print("Learned bias:", learned_bias)
print("Bias error:", bias_error)

print("\nLoss history:", loss_history)

assert loss_history[-1] < loss_history[0]
assert weight_error.abs().max() < 0.2
assert bias_error.abs().max() < 0.2

True weights: tensor([ 2.0000, -3.4000])
Learned weights: tensor([ 2.0004, -3.4004])
Weight error: tensor([-0.0004,  0.0004])

True bias: 4.2
Learned bias: tensor([4.2003])
Bias error: tensor([-0.0003])

Loss history: [9.024181302070618, 0.16023463954031467, 0.002957928909105249, 0.00015280508217983878, 9.872475039446727e-05, 9.749958629254252e-05, 9.729276271536946e-05, 9.730982931796462e-05, 9.724995598662645e-05, 9.764335499494337e-05]
